# eng

Run this notebook from `german/`. Dataset and adapter paths refer to local files.


## Imports

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from peft.peft_model import PeftModel, PeftConfig
from datasets import load_dataset
from torch.utils.data import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from peft import PeftModel, PeftConfig


## Device and precision

In [ ]:
skip_train = True
device = torch.device('cuda:0')
torch_dtype = torch.bfloat16


## Model

In [ ]:
peft_config = PeftConfig.from_pretrained("../english/DeBERTa/results2/checkpoint-22516")
config = AutoConfig.from_pretrained(peft_config.base_model_name_or_path)
base_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    config=config,
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path, use_fast=True)
model = PeftModel.from_pretrained(
    base_model, "../english/DeBERTa/results2/checkpoint-22516",
    is_trainable=not skip_train,
)
model.to(device)
model.eval()


## Datasets

In [ ]:
devset = load_dataset("textdetox/multilingual_toxicity_dataset", split="de")
trainset = load_dataset("csv", data_files="../data/processed_ger_dev.tsv", delimiter="\t")["train"] if not skip_train else None
testset = load_dataset("csv", data_files="../data/eng_processed_test.tsv", delimiter="\t")["train"]

print(trainset)
print(devset)
print(testset)


In [ ]:
class TokenizedDataset(Dataset): #CONFIG
    def __init__(self, dataset, tokenizer):
        self.dataset = dataset
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        text = example["text"]
        if "toxic" in example:
            label = example["toxic"]
        elif "label" in example:
            label = example["label"]
        else:
            label = 0

        if not text:
            text = " "
            label = 0

        tokenized = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokenized["input_ids"].squeeze(0),
            "attention_mask": tokenized["attention_mask"].squeeze(0),
            "label": label
        }


In [ ]:
tokenized_devset = TokenizedDataset(devset, tokenizer)
tokenized_trainset = TokenizedDataset(trainset, tokenizer) if not skip_train else None
tokenized_testset = TokenizedDataset(testset, tokenizer)


## Adapter

In [ ]:
peft_model = model
peft_model.print_trainable_parameters()


## Training and evaluation

In [ ]:
def compute_metrics(pred):
    threshold=0.5
    logits, labels = pred.predictions, pred.label_ids
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = (probs[:, 1] > threshold).astype(int)  # Apply threshold

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [ ]:
training_args = TrainingArguments( #CONFIG
    output_dir=f"./results",
    eval_strategy="epoch",
    learning_rate=1e-5,  # Reduced learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=1e-3,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_trainset,
    eval_dataset=tokenized_devset,
    compute_metrics=compute_metrics
)


In [ ]:
if not skip_train:
    trainer.train()


In [ ]:
checkpoint_dir = "./xlm-t-mix-set/checkpoint-7092"  #CONFIG

peft_model = PeftModel.from_pretrained(base_model, model_id=checkpoint_dir)
trainer.model = peft_model
trainer.args.per_device_eval_batch_size = 16
trainer.compute_metrics = compute_metrics

evaluation_results = trainer.evaluate()
print(evaluation_results)


## Inference

In [ ]:
import pandas as pd
import torch.nn.functional as F


inferenceset = load_dataset("csv", data_files="../data/eng_processed_test.tsv", delimiter="\t")["train"]
tokenized_inferenceset = TokenizedDataset(inferenceset, tokenizer)
predictions_output = trainer.predict(tokenized_inferenceset)

logits = predictions_output.predictions
probs = F.softmax(torch.tensor(logits), dim=1).numpy()

threshold = 0.5
predicted_labels = (probs[:, 1] >= threshold).astype(int)

original_data = pd.read_csv("../data/eng_processed_test.tsv", sep="\t")

df_submission = pd.DataFrame({
    "id": original_data["id"],
    "predicted": predicted_labels
})

df_submission.to_csv("submission_ger.tsv", sep="\t", index=False)
print("Submission file saved as submission_ger.tsv")
